In [ ]:
!pip install plotly

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots  # 👈 導入多圖並聯元件

In [ ]:
df_history= pd.DataFrame({
    'Month': [
        "Month 1", "Month 2", "Month 3", "Month 4", "Month 5", "Month 6", 
        "Month 7", "Month 8", "Month 9", "Month 10", "Month 11", "Month 12"
    ],
    'Actual_Revenue': [
        100000, 102000, 98000, 105000, 110000, 108000, 
        115000, 112000, 120000, 118000, 125000, 122000
    ],
    'Actual_Cost': [
        60000, 61000, 59000, 63000, 65000, 64000, 
        68000, 67000, 72000, 71000, 75000, 74000
    ]
})

In [ ]:
df_history

In [ ]:
rev_std_pct = df_history['Actual_Revenue'].pct_change().std()
cost_std_pct = df_history['Actual_Cost'].pct_change().std()

print("🚀 【CFO 級：營收/成本/淨利 三圖並聯預測系統 7.1】已啟動！")

# 2. 取得經理人連續輸入
user_input = input("\n💬 Please enter multiple REVENUE drift rates (e.g., 0, 0.02, -0.01): ")

if user_input.strip() == "":
    print("👋 No input detected.")
else:
    try:
        drift_list = [float(x.strip()) for x in user_input.split(',') if x.strip() != ""]
        months_future = [f"M+{i}" for i in range(1, 13)]
        
        # 建立一個 1 列 3 行的並聯網頁畫布，橫向排開三大指標
        fig = make_subplots(
            rows=1, cols=3, 
            subplot_titles=("Revenue Forecast", "Cost Forecast", "Net Profit Margin Forecast"),
            horizontal_spacing=0.06 # 子圖之間的間距
        )
        
        # 配色盤 (rgba 格式)
        color_palette = [
            {'main': 'rgba(31,119,180,1)', 'fill': 'rgba(31,119,180,0.05)'},  # 藍
            {'main': 'rgba(44,160,44,1)',  'fill': 'rgba(44,160,44,0.05)'},   # 綠
            {'main': 'rgba(214,39,40,1)',  'fill': 'rgba(214,39,40,0.05)'},   # 紅
            {'main': 'rgba(148,103,189,1)', 'fill': 'rgba(148,103,189,0.05)'} # 紫
        ]
        
        # 3. 循環模擬每一種情境，並將指標並聯塞進對應的子圖中
        for idx, target_drift in enumerate(drift_list):
            np.random.seed(42)
            simulations = 1000
            rev_sims = np.zeros((simulations, 12))
            cost_sims = np.zeros((simulations, 12))
            
            for i in range(simulations):
                cur_rev = 122000   # Month 12 Base
                cur_cost = 74000   # Month 12 Base
                for m in range(12):
                    cur_rev = cur_rev * (1 + np.random.normal(target_drift, rev_std_pct))
                    # 假設成本維持基礎通膨 0.5% 與歷史正常波動
                    cur_cost = cur_cost * (1 + np.random.normal(0.005, cost_std_pct)) 
                    rev_sims[i, m] = cur_rev
                    cost_sims[i, m] = cur_cost
                    
            # 💡 齒輪咬合：淨利由營收宇宙與成本宇宙直接相減得出！
            profit_sims = rev_sims - cost_sims
            
            # 定義一個小工具函數，幫每個子圖算 P10, P50, P90 並畫線（🌟 核心修復：變數名稱底線完全對齊）
            def plot_metrics_to_subplot(sim_data, col_idx, show_legend_flag):
                p10 = np.percentile(sim_data, 10, axis=0)
                p50 = np.percentile(sim_data, 50, axis=0)
                p90 = np.percentile(sim_data, 90, axis=0)
                cp = color_palette[idx % len(color_palette)]
                label_text = f"Drift {target_drift*100:.1f}%"
                
                # 畫 P90 邊界細虛線
                fig.add_trace(go.Scatter(
                    x=months_future, y=p90, line=dict(color=cp['main'], width=0.8, dash='dash'),
                    showlegend=False, hoverinfo='skip'
                ), row=1, col=col_idx)
                
                # 畫 P10 邊界細虛線並渲染陰影
                fig.add_trace(go.Scatter(
                    x=months_future, y=p10, fill='tonexty', fillcolor=cp['fill'],
                    line=dict(color=cp['main'], width=0.8, dash='dash'),
                    name=f"{label_text} (80% Risk Zone)" if show_legend_flag else "",
                    showlegend=show_legend_flag, hoverinfo='skip'
                ), row=1, col=col_idx)
                
                # 畫 P50 主核心線
                fig.add_trace(go.Scatter(
                    x=months_future, y=p50, mode='lines', line=dict(color=cp['main'], width=2.5),
                    name=f"{label_text} (Expected)" if show_legend_flag else "",
                    showlegend=show_legend_flag,
                    hovertemplate=f"<b>{label_text}</b><br>Month: %{{x}}<br>Value: $%{{y:,.0f}}<br><extra></extra>"
                ), row=1, col=col_idx)

            # 分別將營收(欄1)、成本(欄2)、淨利(欄3) 畫進獨立區塊，只讓第三個圖秀圖例，避免重複
            plot_metrics_to_subplot(rev_sims, col_idx=1, show_legend_flag=False)
            plot_metrics_to_subplot(cost_sims, col_idx=2, show_legend_flag=False)
            plot_metrics_to_subplot(profit_sims, col_idx=3, show_legend_flag=True)
            
        # 4. 全域佈局優化與大圖響應式設定
        fig.update_layout(
            title=dict(
                text="Unified Financial Dashboard: Revenue, Cost, & Net Profit Scenario Projections",
                font=dict(size=16, family="Arial", color="black"), x=0.5
            ),
            plot_bgcolor="white",
            hovermode="x unified",
            legend=dict(yanchor="bottom", y=0.01, xanchor="left", x=0.01, bgcolor="rgba(255,255,255,0.8)"),
            margin=dict(l=50, r=30, t=80, b=50),
            height=500
        )
        
        # 格線同步美化
        fig.update_xaxes(title_text="Forecast Horizon (Months)", gridcolor="rgba(0,0,0,0.04)")
        fig.update_yaxes(tickformat="$,.0f", gridcolor="rgba(0,0,0,0.04)")
        
        # 秀出終極三聯圖
        fig.show(config={'scrollZoom': True})
        print("\n🎉 三圖並聯動態交互儀表板渲染成功！(已解除滾輪縮放鎖定)")
        print("💡 經理人判讀密秘：滑鼠滾輪一動，三張圖會『同步伸縮放大』！")
        
    except ValueError:
        print("❌ Input parsing error! Format should be like: 0, 0.02, -0.01")